# 📈 Buy the Dip Detector — Backtest (Classifier)

Notebook ini menjalankan simulasi Walk-Forward Analysis menggunakan model klasifikasi biner.

**Alur:**
1. Model classifier memprediksi apakah hari ini "Buy the Dip" (1) atau bukan (0)
2. Jika sinyal = 1 → Beli di harga Open, TP/SL ditentukan otomatis oleh ATR
3. Exit: Take Profit, Stop Loss, atau Time-Stop (5 hari)

**Prasyarat:** Jalankan `modelling_classifier.ipynb` terlebih dahulu untuk menghasilkan file model.

In [1]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from src.data_loader import load_stock_data
from src.features import create_features
from src.label import create_labels
from src.model_config import FEATURES
from src.backtest import ClassifierBacktester
from src.visualizer import plot_interactive_candlestick

# FEATURES = [
#     'Dist_to_MA20', 'MACD_Hist', 'RSI', 'RSI_3', 'Williams_R', 'distance_to_resistance', 'distance_to_support'
# ]

# ==================== KONFIGURASI ====================
TICKER = 'BUMI.JK'
IN_SAMPLE_END = '2023-12-31'    # Data latih: 2015 - 2023
OOS_START = '2024-01-01'        # Data uji: 2024 - 2025
OOS_END = '2025-12-31'
TODAY = datetime.now().strftime('%Y-%m-%d')
TARGET_NAME = 'is_buy_dip'

# Parameter Backtest
INITIAL_CAPITAL = 5_000_000
MAX_RISK_PCT = 0.02           # Maksimum risiko 2% per trade
MAX_ALLOC_PCT = 0.50          # Maksimum alokasi 50% modal per posisi
ATR_TP_MULTIPLIER = 3.0       # Take Profit = Entry + 2x ATR
ATR_SL_MULTIPLIER = 2.0       # Stop Loss = Entry - 1x ATR
MAX_HOLDING_DAYS = 5          # Time-Stop setelah 5 hari
MIN_BUY_PROBA = 0.2

print(f"Ticker: {TICKER}")
print(f"Modal: Rp{INITIAL_CAPITAL:,.0f}")
print(f"TP/SL: +{ATR_TP_MULTIPLIER}x / -{ATR_SL_MULTIPLIER}x ATR")

Ticker: BUMI.JK
Modal: Rp5,000,000
TP/SL: +3.0x / -2.0x ATR


In [2]:
# ==================== LOAD & PREPARE DATA ====================
df_raw = load_stock_data(ticker=TICKER, start_date='2014-11-01', end_date=TODAY)
df = create_features(df_raw.copy())
df = create_labels(df)

# Split
df_in_sample = df.loc[:IN_SAMPLE_END].copy()
df_oos = df.loc[OOS_START:OOS_END].copy()

# Siapkan X dan y
X_in = df_in_sample[FEATURES].copy()
y_in = df_in_sample[TARGET_NAME].copy()

X_oos = df_oos[FEATURES].copy()
y_oos = df_oos[TARGET_NAME].copy()

print(f"In-Sample  : {len(X_in)} baris")
print(f"Out-of-Sample: {len(X_oos)} baris")

Mengunduh data terbaru untuk BUMI.JK dari Yahoo Finance...
Berhasil memuat 2901 baris data untuk BUMI.JK.
Info Data Cleaning: Menghapus 123 baris data (NaN atau Volume 0).
Berhasil memuat 2778 baris data bersih untuk BUMI.JK.


c:\Users\YOGA\OneDrive\Documents\Repositories\stock-forecasting\notebook\..\src\features.py:218: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  df['ATR_Pct_Change'] = (atr_raw.pct_change(periods=5)).shift(1)


In-Sample  : 2159 baris
Out-of-Sample: 473 baris


In [3]:
# ==================== JALANKAN BACKTEST ====================
backtester = ClassifierBacktester(
    ticker=TICKER,
    initial_capital=INITIAL_CAPITAL,
    max_risk_pct=MAX_RISK_PCT,
    max_alloc_pct=MAX_ALLOC_PCT,
    atr_tp_multiplier=ATR_TP_MULTIPLIER,
    atr_sl_multiplier=ATR_SL_MULTIPLIER,
    max_holding_days=MAX_HOLDING_DAYS,
    min_buy_proba=MIN_BUY_PROBA
)

df_trades, df_equity = backtester.run_backtest(
    X_in=X_in,
    y_in=y_in,
    X_oos=X_oos,
    y_oos=y_oos,
    df_raw_prices=df_raw,
    feature_cols=FEATURES
)

 SIMULASI CLASSIFIER BACKTEST (BUMI.JK) - Buy the Dip Detector
 Modal Awal: Rp5,000,000.00
 TP/SL: +3.0x ATR / -2.0x ATR
--------------------------------------------------
PERIODE: 2024-01 | OOS: 22 Hari Bursa
--------------------------------------------------
[RETRAIN] Bulan pertama: menggunakan model awal tanpa retrain.
--------------------------------------------------
PERIODE: 2024-02 | OOS: 18 Hari Bursa
--------------------------------------------------
[RETRAIN] Melatih ulang classifier dengan 2181 baris data...


c:\Users\YOGA\OneDrive\Documents\Repositories\stock-forecasting\.venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


[RETRAIN] Model classifier berhasil diperbarui.

--------------------------------------------------
PERIODE: 2024-03 | OOS: 18 Hari Bursa
--------------------------------------------------
[RETRAIN] Melatih ulang classifier dengan 2199 baris data...


c:\Users\YOGA\OneDrive\Documents\Repositories\stock-forecasting\.venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


[RETRAIN] Model classifier berhasil diperbarui.

--------------------------------------------------
PERIODE: 2024-04 | OOS: 16 Hari Bursa
--------------------------------------------------
[RETRAIN] Melatih ulang classifier dengan 2217 baris data...


...

[RETRAIN] Model classifier berhasil diperbarui.

  [SELL LOG] 2025-10-01 | Jual (Time-Stop   ) @ Rp161.00  | Lot: 71   | Profit: Rp123,419.30 (+12.14%) | Ekuitas: Rp5,708,668.65
  [BUY LOG]  2025-10-02 | Beli [Uptrend ] @ Rp163.00  | Lot: 55   | Modal : Rp897,844.75            | Ekuitas: Rp5,708,668.65
             [ATR TP/SL] TP: Rp208.00 | SL: Rp143.00
  [SELL LOG] 2025-10-07 | Jual (Stop Loss   ) @ Rp143.00  | Lot: 55   | Profit: Rp-113,311.00 (-12.62%) | Ekuitas: Rp5,595,357.65
  [BUY LOG]  2025-10-08 | Beli [Uptrend ] @ Rp145.00  | Lot: 49   | Modal : Rp711,565.75            | Ekuitas: Rp5,595,357.65
             [ATR TP/SL] TP: Rp194.00 | SL: Rp123.00
  [SELL LOG] 2025-10-15 | Jual (Time-Stop   ) @ Rp136.00  | Lot: 49   | Profit: Rp-46,831.75 ( -6.58%) | Ekuitas: Rp5,548,525.90
  [BUY LOG]  2025-10-31 | Beli [Uptrend ] @ Rp141.00  | Lot: 71   | Modal : Rp1,002,601.65            | Ekuitas: Rp5,548,525.90
             [ATR TP/SL] TP: Rp175.00 | SL: Rp126.00
------------------------

c:\Users\YOGA\OneDrive\Documents\Repositories\stock-forecasting\.venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


[RETRAIN] Model classifier berhasil diperbarui.

  [SELL LOG] 2025-11-07 | Jual (Time-Stop   ) @ Rp141.00  | Lot: 71   | Profit: Rp -4,004.40 ( -0.40%) | Ekuitas: Rp5,544,521.50
  [BUY LOG]  2025-11-10 | Beli [Uptrend ] @ Rp142.00  | Lot: 88   | Modal : Rp1,251,474.40            | Ekuitas: Rp5,544,521.50
             [ATR TP/SL] TP: Rp169.00 | SL: Rp130.00
  [SELL LOG] 2025-11-11 | Jual (Take Profit ) @ Rp169.00  | Lot: 88   | Profit: Rp232,007.60 (+18.54%) | Ekuitas: Rp5,776,529.10
  [BUY LOG]  2025-11-12 | Beli [Uptrend ] @ Rp197.00  | Lot: 44   | Modal : Rp868,100.20            | Ekuitas: Rp5,776,529.10
             [ATR TP/SL] TP: Rp252.00 | SL: Rp172.00
  [SELL LOG] 2025-11-19 | Jual (Time-Stop   ) @ Rp228.00  | Lot: 44   | Profit: Rp132,591.80 (+15.27%) | Ekuitas: Rp5,909,120.90
  [BUY LOG]  2025-11-20 | Beli [Uptrend ] @ Rp230.00  | Lot: 29   | Modal : Rp668,000.50            | Ekuitas: Rp5,909,120.90
             [ATR TP/SL] TP: Rp318.00 | SL: Rp191.00
  [SELL LOG] 2025-11-27 |

c:\Users\YOGA\OneDrive\Documents\Repositories\stock-forecasting\.venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


[RETRAIN] Model classifier berhasil diperbarui.

  [SELL LOG] 2025-12-05 | Jual (Time-Stop   ) @ Rp238.00  | Lot: 30   | Profit: Rp-20,883.00 ( -2.85%) | Ekuitas: Rp5,920,282.90
  [BUY LOG]  2025-12-08 | Beli [Uptrend ] @ Rp240.00  | Lot: 44   | Modal : Rp1,057,584.00            | Ekuitas: Rp5,920,282.90
             [ATR TP/SL] TP: Rp300.00 | SL: Rp214.00
  [SELL LOG] 2025-12-10 | Jual (Take Profit ) @ Rp300.00  | Lot: 44   | Profit: Rp259,116.00 (+24.50%) | Ekuitas: Rp6,179,398.90
  [BUY LOG]  2025-12-11 | Beli [Uptrend ] @ Rp332.00  | Lot: 27   | Modal : Rp897,744.60            | Ekuitas: Rp6,179,398.90
             [ATR TP/SL] TP: Rp430.00 | SL: Rp288.00
  [SELL LOG] 2025-12-18 | Jual (Time-Stop   ) @ Rp340.00  | Lot: 27   | Profit: Rp 17,960.40 ( +2.00%) | Ekuitas: Rp6,197,359.30
  [BUY LOG]  2025-12-19 | Beli [Uptrend ] @ Rp346.00  | Lot: 22   | Modal : Rp762,341.80            | Ekuitas: Rp6,197,359.30
             [ATR TP/SL] TP: Rp466.00 | SL: Rp292.00
  [SELL LOG] 2025-12-30 |

In [4]:
# ==================== DETAIL TRANSAKSI ====================
if df_trades is not None:
    print(f"\nTotal Transaksi: {len(df_trades)}")
    print(f"\n--- Exit Reason Distribution ---")
    print(df_trades['exit_reason'].value_counts())
    print(f"\n--- Ringkasan Profit per Trade ---")
    print(df_trades[['entry_date', 'exit_date', 'entry_price', 'exit_price', 'lots', 'net_profit', 'roi_pct', 'exit_reason', 'days_held']].to_string())
else:
    print("Tidak ada transaksi yang dieksekusi.")


Total Transaksi: 30

--- Exit Reason Distribution ---
exit_reason
Time-Stop      22
Take Profit     5
Stop Loss       3
Name: count, dtype: int64

--- Ringkasan Profit per Trade ---
   entry_date  exit_date  entry_price  exit_price  lots  net_profit    roi_pct  exit_reason  days_held
0  2024-04-26 2024-05-06        103.0       103.0    69    -2842.80  -0.399401    Time-Stop          5
1  2024-05-07 2024-05-15        104.0        92.0    80   -99088.00 -11.891778    Stop Loss          4
2  2024-08-20 2024-08-27         89.0        92.0   117    30847.05   2.957923    Time-Stop          5
3  2024-08-28 2024-09-04         92.0        99.0   105    69452.25   7.178906    Time-Stop          5
4  2024-09-05 2024-09-12        100.0        97.0    96   -32568.00  -3.387419    Time-Stop          5
5  2024-09-13 2024-09-20         98.0       118.0   105   205359.00  19.927252  Take Profit          4
6  2024-09-23 2024-09-30        116.0       122.0    77    42511.70   4.752354    Time-Stop     

In [7]:
# ==================== VISUALISASI CHART INTERAKTIF ====================
# Marker Biru  = Raw Signal dari Classifier (model bilang 'Buy')
# Marker Hijau = Transaksi Beli yang benar-benar dieksekusi
# Marker Merah = Transaksi Jual (TP/SL/Time-Stop)

plot_interactive_candlestick(
    df=df_raw,
    ticker_name=TICKER,
    start_date=OOS_START,
    trades_df=df_trades,
    equity_curve_df=df_equity,
    classifier_predictions_df=backtester.predictions_df
)